# Project 3: Build a News API Client

**Level**: Advanced  
**Skills covered**: `requests`, JSON, OOP, file I/O, `pandas`, `argparse`, exception handling, async (concept)

---

## Project Goal

You will build a command-line news aggregator that fetches real headlines from [NewsAPI](https://newsapi.org/) (free tier).  
If you don't have an API key yet, **the notebook still runs in full** using realistic mock data.

By the end you will have:
- A `NewsClient` class that wraps the API cleanly
- Keyword, category, and language filtering
- Results saved to JSON and CSV via `pandas`
- A `argparse`-based CLI you can run from the terminal
- Robust error handling for rate limits, network failures, and bad keys
- A glimpse of the `async` pattern for concurrent fetching

### Getting a free API key (optional)

1. Go to https://newsapi.org/register
2. Create a free account
3. Copy your API key from the dashboard
4. Paste it below: `API_KEY = "your_key_here"`

The free tier allows 100 requests/day and returns up to 100 articles per request.

---

## Steps Overview

1. Making API requests with `requests`
2. Parsing the JSON response
3. `NewsClient` class
4. Filtering: keyword, category, language
5. Saving to JSON and CSV with `pandas`
6. CLI with `argparse`
7. Error handling
8. Bonus: sentiment analysis teaser

---

## Setup

Install required packages if they are not already present.

In [ ]:
# Install dependencies (run once)
# !pip install requests pandas

# ---- Configuration ----
# Replace with your real API key, or leave as-is to use mock data
API_KEY = "YOUR_API_KEY_HERE"

USE_MOCK = (API_KEY == "YOUR_API_KEY_HERE")

if USE_MOCK:
    print("No API key set — running in MOCK mode. All responses use built-in sample data.")
else:
    print(f"API key configured. Live mode enabled.")

---

## Step 1: Making API Requests

The NewsAPI has two main endpoints:
- `/v2/top-headlines` — breaking news by country/category
- `/v2/everything` — search across millions of articles

In [ ]:
import requests

BASE_URL = "https://newsapi.org/v2"

def fetch_top_headlines(api_key, country="us", category=None, page_size=5):
    """
    Fetch top headlines from NewsAPI.
    Returns the raw JSON dict from the API.
    """
    params = {
        "apiKey": api_key,
        "country": country,
        "pageSize": page_size,
    }
    if category:
        params["category"] = category

    response = requests.get(f"{BASE_URL}/top-headlines", params=params, timeout=10)
    response.raise_for_status()   # raises HTTPError for 4xx/5xx
    return response.json()


def fetch_everything(api_key, query, language="en", page_size=5, sort_by="publishedAt"):
    """
    Search all articles matching a keyword query.
    """
    params = {
        "apiKey": api_key,
        "q": query,
        "language": language,
        "pageSize": page_size,
        "sortBy": sort_by,
    }
    response = requests.get(f"{BASE_URL}/everything", params=params, timeout=10)
    response.raise_for_status()
    return response.json()


print("Request functions defined.")
print("URL structure: GET https://newsapi.org/v2/top-headlines?apiKey=...&country=us")

**Key concepts**: `requests.get()`, query params dict, `timeout`, `raise_for_status()`, `.json()`.

---

## Step 2: Parsing the JSON Response

Understand the API response structure before building the client class.

In [ ]:
# --- Mock response (identical shape to the real API) ---
MOCK_RESPONSE = {
    "status": "ok",
    "totalResults": 5,
    "articles": [
        {
            "source": {"id": "bbc-news", "name": "BBC News"},
            "author": "BBC News",
            "title": "Python 3.13 Released with Major Performance Improvements",
            "description": "The latest Python release delivers significant speed gains via the new free-threaded mode.",
            "url": "https://bbc.co.uk/news/technology/python-3-13",
            "urlToImage": "https://example.com/img/python.jpg",
            "publishedAt": "2024-10-08T10:30:00Z",
            "content": "Python 3.13 has been officially released, bringing experimental support for a GIL-free mode..."
        },
        {
            "source": {"id": "techcrunch", "name": "TechCrunch"},
            "author": "Sarah Chen",
            "title": "OpenAI Announces GPT-5 Preview for Developers",
            "description": "OpenAI made GPT-5 available in preview through its API with enhanced reasoning capabilities.",
            "url": "https://techcrunch.com/2024/gpt5-preview",
            "urlToImage": None,
            "publishedAt": "2024-10-07T14:00:00Z",
            "content": "OpenAI has begun rolling out a preview of GPT-5 to select developers..."
        },
        {
            "source": {"id": "the-verge", "name": "The Verge"},
            "author": "James Park",
            "title": "Rust Overtakes Java in Stack Overflow Developer Survey",
            "description": "For the ninth consecutive year, Rust tops the most-loved language list.",
            "url": "https://theverge.com/rust-survey-2024",
            "urlToImage": "https://example.com/img/rust.jpg",
            "publishedAt": "2024-10-06T09:15:00Z",
            "content": "Rust has once again topped the Stack Overflow annual survey..."
        },
        {
            "source": {"id": None, "name": "Wired"},
            "author": None,
            "title": "The Hidden Cost of AI Data Centers on Global Power Grids",
            "description": "New research quantifies the energy demands of large language model training runs.",
            "url": "https://wired.com/ai-power-grids",
            "urlToImage": "https://example.com/img/ai.jpg",
            "publishedAt": "2024-10-05T11:45:00Z",
            "content": "Researchers at MIT have published a comprehensive analysis of AI energy consumption..."
        },
        {
            "source": {"id": "ars-technica", "name": "Ars Technica"},
            "author": "Tim Lee",
            "title": "Linux Kernel 6.11 Merges First Real-Time Support Patches",
            "description": "After years of effort, real-time patches land in the mainline Linux kernel.",
            "url": "https://arstechnica.com/linux-6-11-rt",
            "urlToImage": None,
            "publishedAt": "2024-10-04T16:20:00Z",
            "content": "The Linux kernel 6.11 has merged the long-awaited PREEMPT_RT patches..."
        },
    ]
}


def parse_articles(response):
    """
    Extract a clean list of article dicts from an API response.
    Normalises None values to empty strings.
    """
    articles = response.get("articles", [])
    cleaned = []
    for art in articles:
        cleaned.append({
            "title":        art.get("title") or "",
            "description": art.get("description") or "",
            "source":       art.get("source", {}).get("name") or "Unknown",
            "author":       art.get("author") or "Unknown",
            "published_at": art.get("publishedAt") or "",
            "url":          art.get("url") or "",
            "content":      art.get("content") or "",
        })
    return cleaned


articles = parse_articles(MOCK_RESPONSE)
print(f"Parsed {len(articles)} articles\n")
for a in articles:
    print(f"  [{a['source']}] {a['title']}")
    print(f"    {a['published_at']}  —  {a['url']}")
    print()

**Key concepts**: `dict.get()` with defaults, nested dict access, normalising `None` with `or`.

---

## Step 3: The `NewsClient` Class

Wrap the raw request functions in a class that handles auth, switching between mock/live, and caching.

In [ ]:
import requests
from datetime import datetime


class NewsAPIError(Exception):
    """Raised for NewsAPI-specific errors (bad key, rate limit, etc.)."""
    pass


class NewsClient:
    """
    A clean wrapper around the NewsAPI.

    Instantiate with your API key. If use_mock=True, returns sample data
    without making any network calls — useful for development and testing.
    """

    BASE_URL = "https://newsapi.org/v2"

    VALID_CATEGORIES = [
        "business", "entertainment", "general",
        "health", "science", "sports", "technology",
    ]

    VALID_LANGUAGES = [
        "ar", "de", "en", "es", "fr", "he",
        "it", "nl", "no", "pt", "ru", "sv", "ud", "zh",
    ]

    def __init__(self, api_key, use_mock=False):
        self.api_key = api_key
        self.use_mock = use_mock
        self.session = requests.Session()
        self.session.headers.update({"X-Api-Key": api_key})
        self._last_response = None

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _get(self, endpoint, params):
        """Make a GET request; raise NewsAPIError on non-200 responses."""
        if self.use_mock:
            return MOCK_RESPONSE

        url = f"{self.BASE_URL}/{endpoint}"
        try:
            resp = self.session.get(url, params=params, timeout=10)
        except requests.ConnectionError:
            raise NewsAPIError("No internet connection or DNS failure.")
        except requests.Timeout:
            raise NewsAPIError("Request timed out after 10 seconds.")

        data = resp.json()

        if resp.status_code == 401:
            raise NewsAPIError("Invalid API key. Check your key at newsapi.org.")
        if resp.status_code == 429:
            raise NewsAPIError("Rate limit exceeded. Free tier: 100 requests/day.")
        if resp.status_code != 200 or data.get("status") != "ok":
            msg = data.get("message", "Unknown error")
            raise NewsAPIError(f"API error {resp.status_code}: {msg}")

        self._last_response = data
        return data

    @staticmethod
    def _parse(response):
        return parse_articles(response)

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def top_headlines(self, country="us", category=None, page_size=10):
        """
        Fetch top headlines.

        Returns a list of cleaned article dicts.
        """
        if category and category not in self.VALID_CATEGORIES:
            raise ValueError(
                f"Invalid category '{category}'. Choose from: {self.VALID_CATEGORIES}"
            )
        params = {"country": country, "pageSize": min(page_size, 100)}
        if category:
            params["category"] = category
        return self._parse(self._get("top-headlines", params))

    def search(self, query, language="en", sort_by="publishedAt", page_size=10):
        """
        Search all articles for a keyword.

        sort_by: 'relevancy' | 'popularity' | 'publishedAt'
        """
        if not query.strip():
            raise ValueError("Search query cannot be empty.")
        params = {
            "q": query,
            "language": language,
            "sortBy": sort_by,
            "pageSize": min(page_size, 100),
        }
        return self._parse(self._get("everything", params))

    def sources(self, category=None, language="en", country="us"):
        """Return available news sources."""
        if self.use_mock:
            return [{"id": "bbc-news", "name": "BBC News"},
                    {"id": "techcrunch", "name": "TechCrunch"}]
        params = {"language": language, "country": country}
        if category:
            params["category"] = category
        resp = self._get("top-headlines/sources", params)
        return resp.get("sources", [])


# Create the client
client = NewsClient(api_key=API_KEY, use_mock=USE_MOCK)

# Fetch headlines
articles = client.top_headlines(country="us", page_size=5)
print(f"Fetched {len(articles)} articles\n")
for a in articles[:3]:
    print(f"  {a['title']}")
    print(f"  Source: {a['source']} | {a['published_at']}")
    print()

**Key concepts**: `requests.Session`, custom exceptions, `@staticmethod`, kwargs with defaults.

---

## Step 4: Filtering by Keyword, Category, and Language

In [ ]:
# The API handles category/language filtering server-side.
# We can also do client-side keyword filtering on the returned articles.

def filter_articles(articles, keyword=None, min_length=0):
    """
    Filter a list of article dicts.

    keyword    : if set, only keep articles where keyword appears in title or description
    min_length : minimum character length of description
    """
    result = articles

    if keyword:
        kw = keyword.lower()
        result = [
            a for a in result
            if kw in a["title"].lower() or kw in a["description"].lower()
        ]

    if min_length > 0:
        result = [a for a in result if len(a["description"]) >= min_length]

    return result


# Demo
all_articles = client.top_headlines(page_size=10)
print(f"Total articles fetched: {len(all_articles)}")

# Filter by keyword
python_articles = filter_articles(all_articles, keyword="python")
print(f"Articles mentioning 'python': {len(python_articles)}")
for a in python_articles:
    print(f"  - {a['title']}")

# Filter by description length (skip stub articles)
long_articles = filter_articles(all_articles, min_length=50)
print(f"\nArticles with description >= 50 chars: {len(long_articles)}")

In [ ]:
# Use the API's own category and language parameters

tech_news = client.top_headlines(country="us", category="technology", page_size=5)
print(f"Technology headlines: {len(tech_news)}")
for a in tech_news:
    print(f"  [{a['source']}] {a['title']}")

# Search with language filter
search_results = client.search("artificial intelligence", language="en", page_size=3)
print(f"\nSearch results for 'artificial intelligence': {len(search_results)}")
for a in search_results:
    print(f"  {a['title']}")

---

## Step 5: Saving to JSON and CSV with pandas

In [ ]:
import json
import os

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False
    print("pandas not installed. Run: pip install pandas")
    print("JSON saving will still work.")


def save_to_json(articles, filepath="news_articles.json", append=False):
    """
    Save articles to a JSON file.
    append=True merges with existing data (deduplicates by URL).
    """
    existing = []
    if append and os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            existing = json.load(f)

    # Deduplicate by URL
    seen_urls = {a["url"] for a in existing}
    new_articles = [a for a in articles if a["url"] not in seen_urls]
    combined = existing + new_articles

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(new_articles)} new articles ({len(combined)} total) to '{filepath}'")
    return filepath


def save_to_csv(articles, filepath="news_articles.csv"):
    """Save articles to a CSV file using pandas."""
    if not PANDAS_AVAILABLE:
        print("pandas not available — skipping CSV save.")
        return None

    df = pd.DataFrame(articles)

    # Parse dates if possible
    try:
        df["published_at"] = pd.to_datetime(df["published_at"])
    except Exception:
        pass

    df.to_csv(filepath, index=False, encoding="utf-8")
    print(f"Saved {len(df)} articles to '{filepath}'")
    return filepath


def load_from_csv(filepath="news_articles.csv"):
    """Load saved articles from CSV into a DataFrame."""
    if not PANDAS_AVAILABLE:
        raise ImportError("pandas is required to load CSV files.")
    return pd.read_csv(filepath, parse_dates=["published_at"])


# Demo
articles = client.top_headlines(page_size=5)
save_to_json(articles, "news_articles.json")

if PANDAS_AVAILABLE:
    save_to_csv(articles, "news_articles.csv")
    df = load_from_csv("news_articles.csv")
    print(f"\nDataFrame shape: {df.shape}")
    print(df[["title", "source", "published_at"]].to_string(index=False))

In [ ]:
# Pandas analysis: most prolific sources
if PANDAS_AVAILABLE:
    df = load_from_csv("news_articles.csv")
    print("Articles per source:")
    print(df["source"].value_counts().to_string())

    print("\nDescriptions sorted by length (longest first):")
    df["desc_length"] = df["description"].str.len()
    print(df.sort_values("desc_length", ascending=False)[["title", "desc_length"]].to_string(index=False))

**Key concepts**: `json.dump/load`, deduplication with sets, `pd.DataFrame`, `pd.to_datetime`, `value_counts`.

---

## Step 6: CLI with argparse

Write the script so it can be run from the terminal with flags.

In [ ]:
# This cell shows the structure of the CLI script.
# Save it as news_cli.py and run:
#   python news_cli.py top-headlines --country us --category technology
#   python news_cli.py search --query "Python" --save

CLI_SCRIPT = '''
#!/usr/bin/env python3
"""
news_cli.py — Command-line news aggregator using NewsAPI.

Usage examples:
  python news_cli.py top-headlines
  python news_cli.py top-headlines --country gb --category technology --n 10
  python news_cli.py search --query "machine learning" --save
  python news_cli.py search --query "Python" --lang en --sort popularity
"""
import argparse
import os
import sys

# --- Paste your NewsClient class here or import it ---
# from news_client import NewsClient, save_to_json, save_to_csv

API_KEY = os.environ.get("NEWS_API_KEY", "YOUR_API_KEY_HERE")
USE_MOCK = API_KEY == "YOUR_API_KEY_HERE"


def build_parser():
    parser = argparse.ArgumentParser(
        description="Fetch news headlines from NewsAPI.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )
    subparsers = parser.add_subparsers(dest="command", required=True)

    # --- top-headlines subcommand ---
    hl = subparsers.add_parser("top-headlines", help="Fetch top headlines.")
    hl.add_argument("--country",  default="us",  help="Country code (default: us)")
    hl.add_argument("--category", default=None,  help="Category (technology, sports, ...)")
    hl.add_argument("--n",        type=int, default=5, help="Number of articles (default: 5)")
    hl.add_argument("--save",     action="store_true", help="Save results to JSON and CSV")

    # --- search subcommand ---
    se = subparsers.add_parser("search", help="Search all articles.")
    se.add_argument("--query",  required=True,  help="Search keyword")
    se.add_argument("--lang",   default="en",   help="Language code (default: en)")
    se.add_argument("--sort",   default="publishedAt",
                    choices=["relevancy", "popularity", "publishedAt"],
                    help="Sort order")
    se.add_argument("--n",      type=int, default=5, help="Number of articles")
    se.add_argument("--save",   action="store_true")

    return parser


def print_articles(articles):
    for i, a in enumerate(articles, 1):
        print(f"\n{i}. {a[\'title\']}")
        print(f"   Source: {a[\'source\']} | {a[\'published_at\']}")
        if a[\'description\']:
            print(f"   {a[\'description\']}")
        print(f"   {a[\'url\'][:80]}")


def main():
    parser = build_parser()
    args = parser.parse_args()

    client = NewsClient(API_KEY, use_mock=USE_MOCK)

    try:
        if args.command == "top-headlines":
            articles = client.top_headlines(
                country=args.country,
                category=args.category,
                page_size=args.n,
            )
        else:  # search
            articles = client.search(
                query=args.query,
                language=args.lang,
                sort_by=args.sort,
                page_size=args.n,
            )
    except NewsAPIError as e:
        print(f"Error: {e}", file=sys.stderr)
        sys.exit(1)

    print_articles(articles)

    if args.save:
        save_to_json(articles)
        save_to_csv(articles)


if __name__ == "__main__":
    main()
'''

print(CLI_SCRIPT)

In [ ]:
# Demonstrate argparse in-notebook by simulating argv
import argparse
import sys

def build_parser():
    parser = argparse.ArgumentParser(description="News CLI demo")
    subparsers = parser.add_subparsers(dest="command", required=True)

    hl = subparsers.add_parser("top-headlines")
    hl.add_argument("--country",  default="us")
    hl.add_argument("--category", default=None)
    hl.add_argument("--n",        type=int, default=5)
    hl.add_argument("--save",     action="store_true")

    se = subparsers.add_parser("search")
    se.add_argument("--query",  required=True)
    se.add_argument("--lang",   default="en")
    se.add_argument("--n",      type=int, default=5)
    se.add_argument("--save",   action="store_true")

    return parser


parser = build_parser()

# Simulate: python news_cli.py top-headlines --country us --n 3
args = parser.parse_args(["top-headlines", "--country", "us", "--n", "3"])
print(f"command  : {args.command}")
print(f"country  : {args.country}")
print(f"n        : {args.n}")
print(f"save     : {args.save}")

**Key concepts**: `argparse.ArgumentParser`, subparsers, `add_argument`, `action="store_true"`, `parse_args(list)`.

---

## Step 7: Error Handling

Production clients must handle every failure mode gracefully.

In [ ]:
import time

class RetryingNewsClient(NewsClient):
    """
    Extends NewsClient with automatic retries and exponential backoff.
    """

    def __init__(self, api_key, use_mock=False, max_retries=3, backoff_factor=2):
        super().__init__(api_key, use_mock)
        self.max_retries = max_retries
        self.backoff_factor = backoff_factor

    def _get(self, endpoint, params):
        last_error = None
        for attempt in range(1, self.max_retries + 1):
            try:
                return super()._get(endpoint, params)
            except NewsAPIError as e:
                last_error = e
                error_str = str(e)
                # Don't retry auth errors — they won't resolve on retry
                if "Invalid API key" in error_str or "Rate limit" in error_str:
                    raise
                wait = self.backoff_factor ** attempt
                print(f"  Attempt {attempt}/{self.max_retries} failed: {e}")
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
        raise last_error


# Error handling demonstration (all errors shown without network calls)
error_scenarios = [
    (401, "apiKeyInvalid",  "Your API key is invalid."),
    (429, "rateLimited",    "You have made too many requests recently."),
    (500, "serverError",    "Internal server error."),
    (200, "ok",             None),  # success
]

print("Error handling matrix:")
print(f"  {'Status':<8} {'Code':<20} {'Action'}")
print("-" * 55)
for status, code, msg in error_scenarios:
    if status == 401:
        action = "Raise NewsAPIError — do not retry (fix key)"
    elif status == 429:
        action = "Raise NewsAPIError — wait and retry"
    elif status >= 500:
        action = "Raise NewsAPIError — retry with backoff"
    else:
        action = "Return parsed articles"
    print(f"  {status:<8} {code:<20} {action}")

In [ ]:
# Validate all possible user input errors before hitting the API

def validate_request(command, **kwargs):
    """
    Validate parameters before making any API call.
    Returns (is_valid: bool, error_message: str | None).
    """
    VALID_CATEGORIES = ["business", "entertainment", "general",
                        "health", "science", "sports", "technology"]
    VALID_SORT = ["relevancy", "popularity", "publishedAt"]

    if command == "search":
        query = kwargs.get("query", "")
        if not query or not query.strip():
            return False, "Search query cannot be empty."
        if len(query) > 500:
            return False, "Query too long (max 500 characters)."
        sort = kwargs.get("sort_by", "publishedAt")
        if sort not in VALID_SORT:
            return False, f"Invalid sort_by '{sort}'. Choose from {VALID_SORT}."

    if command == "top-headlines":
        category = kwargs.get("category")
        if category and category not in VALID_CATEGORIES:
            return False, f"Invalid category '{category}'. Choose from {VALID_CATEGORIES}."

    n = kwargs.get("page_size", 5)
    if not isinstance(n, int) or n < 1 or n > 100:
        return False, "page_size must be between 1 and 100."

    return True, None


# Test validation
test_cases = [
    ("search",        {"query": "Python", "page_size": 5}),
    ("search",        {"query": "", "page_size": 5}),
    ("search",        {"query": "AI", "sort_by": "newest", "page_size": 5}),
    ("top-headlines", {"category": "technology", "page_size": 10}),
    ("top-headlines", {"category": "cooking",    "page_size": 10}),
    ("top-headlines", {"page_size": 200}),
]

for cmd, kwargs in test_cases:
    valid, err = validate_request(cmd, **kwargs)
    status = "OK " if valid else "ERR"
    print(f"  [{status}] {cmd}({kwargs}) -> {err or 'valid'}")

---

## Step 8: Bonus — Sentiment Analysis Teaser

The pattern for adding sentiment scoring to headlines — no extra install required for the demo.

In [ ]:
# --- Option A: Zero-dependency keyword approach (works right now) ---

POSITIVE_WORDS = {
    "breakthrough", "improvement", "growth", "success", "launch",
    "record", "innovation", "wins", "soars", "booms", "rising",
}
NEGATIVE_WORDS = {
    "crash", "fail", "ban", "risk", "threat", "warning", "decline",
    "layoffs", "drops", "plunges", "worst", "crisis", "hack",
}

def simple_sentiment(text):
    """
    Very naive keyword-based sentiment.
    Returns 'positive', 'negative', or 'neutral'.
    """
    words = set(text.lower().split())
    pos_hits = words & POSITIVE_WORDS
    neg_hits = words & NEGATIVE_WORDS
    if len(pos_hits) > len(neg_hits):
        return "positive", pos_hits
    if len(neg_hits) > len(pos_hits):
        return "negative", neg_hits
    return "neutral", set()


articles = client.top_headlines(page_size=5)
print("Naive sentiment analysis on headlines:\n")
for a in articles:
    sentiment, hits = simple_sentiment(a["title"])
    marker = {"positive": "(+)", "negative": "(-)", "neutral": "( )"}[sentiment]
    print(f"  {marker} {a['title']}")
    if hits:
        print(f"      keywords: {hits}")

In [ ]:
# --- Option B: TextBlob (pip install textblob) ---

TEXTBLOB_EXAMPLE = """
# Install: pip install textblob
from textblob import TextBlob

for article in articles:
    blob = TextBlob(article['title'])
    polarity = blob.sentiment.polarity        # -1.0 (negative) to +1.0 (positive)
    subjectivity = blob.sentiment.subjectivity  # 0.0 (objective) to 1.0 (subjective)

    label = 'positive' if polarity > 0.1 else ('negative' if polarity < -0.1 else 'neutral')
    print(f'  [{label:8}] polarity={polarity:+.2f}  {article["title"]}')
"""

# --- Option C: transformers (pip install transformers torch) ---

TRANSFORMERS_EXAMPLE = """
# Install: pip install transformers torch
from transformers import pipeline

classifier = pipeline('sentiment-analysis')   # downloads ~67MB model once

titles = [a['title'] for a in articles]
results = classifier(titles)

for title, result in zip(titles, results):
    print(f"  [{result['label']:8}] score={result['score']:.2f}  {title}")
"""

print("TextBlob approach (lexicon-based, fast, offline):")
print(TEXTBLOB_EXAMPLE)

print("Transformers approach (BERT-based, high accuracy, requires GPU-optional install):")
print(TRANSFORMERS_EXAMPLE)

**Key concepts**: set intersection (`&`), progressive enhancement (add libraries when needed), pipeline pattern.

---

## Bonus: Async Version Concept

Fetching multiple endpoints concurrently with `asyncio` and `aiohttp`.

In [ ]:
# --- Concept: fetching multiple categories concurrently ---
# Install: pip install aiohttp

ASYNC_EXAMPLE = '''
import asyncio
import aiohttp

API_KEY = "your_key_here"
CATEGORIES = ["technology", "science", "health", "sports"]

async def fetch_category(session, category):
    url = "https://newsapi.org/v2/top-headlines"
    params = {"apiKey": API_KEY, "country": "us", "category": category, "pageSize": 5}
    async with session.get(url, params=params) as resp:
        data = await resp.json()
        return category, data.get("articles", [])


async def fetch_all_categories():
    async with aiohttp.ClientSession() as session:
        # Launch all requests concurrently
        tasks = [fetch_category(session, cat) for cat in CATEGORIES]
        results = await asyncio.gather(*tasks)

    for category, articles in results:
        print(f"\n{category.upper()} ({len(articles)} articles)")
        for a in articles[:2]:
            print(f"  - {a[\'title\']}")


# In a script: asyncio.run(fetch_all_categories())
# In Jupyter:  await fetch_all_categories()  (Python 3.7+)
'''

print("Async pattern: fetch 4 categories in parallel instead of sequentially")
print()
print("Sequential (sync) time: ~4 * 300ms = ~1.2s")
print("Concurrent (async) time: ~1 * 300ms = ~0.3s  (4x faster)")
print()
print(ASYNC_EXAMPLE)

---

## Summary

| Step | What you built | Skills |
|------|---------------|--------|
| 1 | Raw API requests | `requests.get`, params, `raise_for_status` |
| 2 | JSON parsing | `dict.get`, normalising None, nested access |
| 3 | `NewsClient` class | OOP, `Session`, custom exceptions |
| 4 | Filtering | list comprehensions, API params |
| 5 | JSON + CSV export | `json`, `pandas`, deduplication |
| 6 | CLI | `argparse`, subparsers, `sys.exit` |
| 7 | Error handling | retries, backoff, validation |
| 8 | Sentiment | keyword matching, TextBlob, transformers |
| Bonus | Async | `asyncio`, `aiohttp`, `gather` |

---

## How to Extend This

- **Scheduling**: use `schedule` or `APScheduler` to fetch headlines every hour and append to CSV
- **Alerts**: send an email or Slack message when a keyword appears using `smtplib` or a webhook
- **Dashboard**: build a `streamlit` web app that visualises article counts over time from the CSV
- **RSS fallback**: add a `feedparser`-based backend for sources not on NewsAPI
- **Database**: replace the JSON file with a SQLite database via `sqlite3` or `SQLAlchemy`
- **Full sentiment pipeline**: fine-tune a BERT model on headlines with known sentiment labels